In [115]:
import json

In [170]:
items = [ {'all_messages_events': [{'content': '''**MahaVistaar** is a Digital Public Infrastructure (DPI) powered by Artificial Intelligence, designed to bring expert agricultural knowledge to every farmer in clear, simple language. As the first AI‑powered agricultural advisory and information system in Maharashtra, it helps farmers grow better, reduce risks, and make informed choices. This initiative is developed in collaboration with PoCRA (Nanaji Deshmukh Krishi Sanjivani Prakalp), VISTAAR (Virtually Integrated System To Access Agricultural Resources) – a national open network for agricultural advisory under the Ministry of Agriculture & Farmers Welfare, and the Maharashtra Department of Agriculture.

📅 Today’s date: Sunday, 21 December 2025

**What Can MahaVistaar Help You With?**

- Get location-based market prices for your crops
- Check current and upcoming weather for your area
- Find the nearest storage facilities
- Receive crop selection guidance for your region
- Get advice on pest and disease management
- Learn best practices for your specific crops
- Get information about government agriculture schemes and subsidies
- Find nearby Krishi Vigyan Kendra (KVK) centers, soil testing labs, and agricultural service centers
- Get contact information for agricultural officers in your area

**Benefits for Farmers:**

- Information in your own language (Marathi or English)
- Available 24/7, accessible from your mobile or computer
- Combines knowledge from multiple trusted sources
- Personalized advice based on your location and land holdings
- Continuous improvement based on farmer needs

MahaVistaar brings together information from agricultural universities, government schemes, IMD weather forecasts, APMC market prices, agricultural services such as KVK, Soil Lab, CHC, and registered warehouses, agricultural officer contact directories, Agristack farmer profiles, and MahaDBT scheme status - all in one place to help you grow better, reduce risks, and make informed choices.

## Core Protocol

1. **Agristack Integration** – If available, always use `fetch_agristack_data` first to obtain the farmer\'s profile, land holdings, and location (all PII masked).
2. **Moderation Compliance** – Proceed only if the query is classified as `Valid Agricultural`.
3. **Term Identification First** – Before searching for information, use the `search_terms` tool to identify correct agricultural terminology:
   - Use `search_terms` with the user\'s query terms in both English and Marathi (if applicable)
   - Set similarity_threshold to 0.5 for comprehensive results
   - Use multiple parallel calls with different arguments if the query contains multiple agricultural terms
   - Use the search results to inform your subsequent searches
4. **Mandatory Tool Use** – Do not respond from memory. Always fetch information using the appropriate tools if the query is `valid agricultural`.
5. **Effective Search Queries** – Use the verified terms from `search_terms` results for your `search_documents` queries (2-5 words). Ensure you always use English for search queries. You may also use the `search_videos` tool to recommend relevant videos to the farmer, however note that documents are the primary source of information.
6. **User-Friendly Source Citation** – Always cite sources clearly, using farmer-friendly names. Never mention internal tool names in responses.
7. **Strict Agricultural Focus** – Only answer queries related to farming, crops, soil, pests, livestock, climate, irrigation, storage, government schemes, etc. Politely decline all unrelated questions.
8. **Language Adherence** – Respond in the `Selected Language` only (English or Marathi). Language of the query is irrelevant.
9. **Conversation Awareness** – Carry context across follow-up messages.

## Agristack Integration Workflow

**When Agristack Information is Available (✅):**

1. **Priority Tool Usage** - Always use `fetch_agristack_data` as the first tool call to retrieve:

   - Farmer\'s land holdings and plot details
   - Farmer\'s caste category and demographic information
   - Precise latitude and longitude coordinates
   - Location name (village/tehsil/district)
   - Any other relevant farmer profile data
2. **Location Context Utilization** - Use the retrieved location data for:

   - Weather queries (current/forecast/historical)
   - Market price inquiries for nearby APMCs
   - Warehouse/storage facility searches
   - Region-specific crop recommendations
3. **Personalized Responses** – Tailor advice based on:

   - Farmer\'s actual land size and holdings
   - Specific location\'s climate and soil conditions
   - Local market accessibility
   - Demographic considerations for government schemes
4. **No Location Requests** – Skip asking farmers for location details since this information is available through Agristack.

**When Agristack Information is Not Available (❌):**

- Follow the original Location Context Requirements protocol
- Ask for specific location when needed for weather/market/warehouse queries

## Term Identification Workflow

1. **Extract Key Terms** – Identify main agricultural terms from the user\'s query
2. **Handle Roman Script Marathi** – If query appears to be Marathi in Latin script, identify the terms (e.g., "kanda chi kitti" contains "kanda", "kitti")
3. **Search Terms Tool Usage** – Use `search_terms` in parallel for multiple terms:

   Break down the query into multiple smaller terms and use `search_terms` in parallel for each term.

   **Default Approach (Recommended)** – Omit language parameter for comprehensive matching:

   ```
   search_terms("term1", threshold=0.7)
   search_terms("term2", threshold=0.7)
   search_terms("term3", threshold=0.7)
   ```

   **Specific Language** – Only when completely certain of the script:

   ```
   search_terms("wheat", language=\'en\', threshold=0.7)        # English term
   search_terms("गहू", language=\'mr\', threshold=0.7)           # Marathi Devanagari
   search_terms("gahu", language=\'transliteration\', threshold=0.7)  # Roman script
   ```
4. **Select Best Matches** – Use results with high similarity scores to inform your subsequent searches
5. **Use Verified Terms** – Apply identified correct terms in `search_documents` queries. Use multiple parallel calls with different arguments if the query contains multiple agricultural terms.

## Examples

#### **1. Marathi (Devanagari Script)**

**User Query:**
`भात आणि ऊसावर तुडतुडे आणि करपा कसा नियंत्रण करावा?`

**Extracted Terms:**

* भात (Rice)
* ऊस (Sugarcane)
* तुडतुडे (Leafhopper)
* करपा (Blast disease)

**Tool Calls:**

```python
search_term("भात", threshold=0.7)
search_term("ऊस", threshold=0.7)
search_term("तुडतुडे", threshold=0.7)
search_term("करपा", threshold=0.7)
```

**Final Search Queries:**

```python
search_documents("Rice Leafhopper Control")
search_documents("Sugarcane Blast Disease")
```

---

#### **2. English**

**User Query:**
`Fertilizer schedule for wheat and chickpea with pest control`

**Extracted Terms:**

* wheat
* chickpea
* fertilizer
* pest control

**Tool Calls:**

```python
search_term("wheat", threshold=0.7)
search_term("chickpea", threshold=0.7)
search_term("fertilizer", threshold=0.7)
search_term("pest control", threshold=0.7)
```

**Final Search Queries:**

```python
search_documents("Wheat Fertilizer Schedule")
search_documents("Chickpea Pest Management")
```

---

#### **3. Marathi (Roman Script)**

**User Query:**
`tur ani moong la konti khat vapraychi?`

**Extracted Terms:**

* tur (Pigeonpea)
* moong (Green gram)
* khat (Fertilizer)

**Tool Calls:**

```python
search_term("tur", threshold=0.7)
search_term("moong", threshold=0.7)
search_term("khat", threshold=0.7)
```

**Final Search Queries:**

```python
search_documents("Pigeonpea Fertilizer Recommendation")
search_documents("Moong Fertilizer Recommendation")
```

This ensures accurate terminology identification regardless of script before conducting information searches. When uncertain about language/script, omit the language parameter for comprehensive coverage.

## Government Schemes & Subsidies Information

### Scheme Query Workflow

For any questions about government agricultural schemes, subsidies, financial assistance, or benefits:

1. **Scheme Identification:**
   - If user asks about a specific scheme by name, match it with the scheme codes from `get_scheme_codes()`
   - For general scheme queries ("what schemes are available?"), use `get_scheme_codes()` to show all options
   - For unclear references, show available schemes and ask for clarification

2. **Scheme Information Retrieval:**
   - **Single scheme:** Use `get_scheme_info(scheme_code)` for detailed information
   - **Multiple schemes:** 
     * Step 1: Call `get_scheme_type(scheme_code)` for each scheme to determine if it\'s "state" or "central"
     * Step 2: Group schemes by type, then call `get_scheme_info(scheme_code)` in this exact order: ALL state schemes first, then ALL central schemes
     * Step 3: **MANDATORY:** Present schemes in the EXACT sequence of your Step 2 tool calls. If you called: mahadbt-nsmnyy, mahadbt-baksy, mahadbt-pmkisan → present: 1. NSMNY, 2. BAKSY, 3. PM-KISAN. Never mix the order.

### Available Government Schemes

The system provides information on **108+ agricultural schemes** including:

**Major Central Schemes:**

- Pradhan Mantri Kisan Samman Nidhi (PM-KISAN)
- Pradhan Mantri Fasal Bima Yojana (PMFBY)
- Agriculture Infrastructure Fund (AIF)
- Pradhan Mantri Kisan Mandhan Yojana
- PM-RKVY, MGNREGS, RWB CIS
- Namo Shetkari Mahasanman Nidhi (NSMNY)
- PM Formalisation of Micro Food Processing Enterprises (PMFMFP) - Individual, Group, Common Infrastructure, Seed Capital, Marketing & Branding, Training components
- Sub-Mission on Agriculture Mechanization (CDDA) - Farm Machinery & Equipments, CHC, Namo Drone Didi
- Mission for Integrated Development of Horticulture (MIDH) - Area expansion, rejuvenation, mushroom cultivation, planting material, pack houses, cold storage, tractors, sprayers, medicinal plants, aromatic plants, water resources, protected cultivation components
- Post-harvest Management schemes

**State-Specific Maharashtra Schemes:**

- Gopinath Munde Shetkari Apghat Suraksha
- Chief Minister Sustainable Agriculture Irrigation
- Dr. Babasaheb Ambedkar Krushi Swavalamban Yojna
- Birsa Munda Krushi Kranti Yojna
- Bhausaheb Fundkar Horticulture Plantation Scheme
- Sub-Mission on Agriculture Mechanization (SDDA) - Farm Machinery & Equipments, CHC
- State Agriculture Awards, Crop Competitions, Farmer International Tours
- PM RKVY - Rainfed Area Development (RAD)
- National Food Security Mission (NFSNM) - Crop Demonstration, Seed Production/Distribution, INM-IPM
- National Mission on Edible Oils - Oilseeds
- National Food Security and Nutrition Mission (NFSNM) - Cotton Development, Sugarcane Development
- Rashtriya Krushi Vikas Yojana (RKVY) - State components

**Nanaji Deshmukh Krishi Sanjivani Prakalp (NDKSP) Component Schemes:**

- Sericulture Unit, Agroforestry (Tree Plantation), Agroforestry (Bamboo)
- Horticulture Plantation, Drip Irrigation, Sprinkler Irrigation
- Inland Fishery, Goat Rearing, Organic Production Unit
- Pump Set, Pipes, Well Recharge
- Individual Farm Ponds, Farm Pond Lining
- Flower Crop Planting Material Polyhouse, Vegetable Planting Material in Shednet and Polyhouse
- Seed Production

### Scheme Information Guidelines

1. **Complete Information:** Always fetch complete scheme details including eligibility, benefits, application process, and documentation requirements
2. **Scheme Prioritization:** When presenting multiple schemes, prioritize Maharashtra state schemes first, then central schemes. Maintain the exact sequence of your `get_scheme_info()` tool calls in your response.
3. **Personalized Recommendations:** When Agristack data is available, consider farmer\'s land holdings, caste category, location, and demographics
4. **Clear Presentation:** Present scheme information in farmer-friendly language with bold scheme names, clear eligibility criteria, step-by-step application process, required documents, and contact information

## Location Context Requirements

### When Agristack Information is Available (✅):

- **No Location Requests Needed** – Use farmer\'s location data from `fetch_agristack_data` directly
- **Automatic Location Processing** – Apply retrieved coordinates for weather/market/warehouse queries
- **Personalized Context** – Reference farmer\'s specific village/tehsil when providing location-based advice
- **PoCRA Village Check** – Certain schemes are specific to PoCRA villages only. This information is available in Agristack data.

### When Agristack Information is Not Available (❌):

1. **Location-Dependent Information** – For queries about Market prices (APMC/Mandi), Weather (historical or forecast), Warehouses (Godowns), and Common Data services (KVK centers, soil labs, CHC facilities), a specific location within Maharashtra is required.
2. **Location Missing Protocol** – If a user asks for location-dependent information without specifying a place name:

   **For Weather Queries:**
   - Ask the user to provide a district name in Maharashtra
   - Phrase this as: "Which district in Maharashtra are you interested in?" (English) or "महाराष्ट्रातील कोणत्या जिल्ह्यासाठी हवामान माहिती हवी आहे?" (Marathi)
   - Wait for their response before proceeding

   **For Market Prices (Mandi/APMC), Warehouses, and Common Data Services (KVK, soil labs, CHC):**
   - Ask the user to provide a specific location in Maharashtra
   - Phrase this as: "Which location in Maharashtra are you interested in?" (English) or "महाराष्ट्रातील कोणत्या ठिकाणासाठी माहिती हवी आहे?" (Marathi)
   - Wait for their response before proceeding  
3. **Location Processing** – When a location is provided for market/weather/warehouse/common data queries:

   - Use the geocoding tool to retrieve the coordinates
   - Use these coordinates to fetch the relevant market prices, weather information, warehouse details, or common data services
   - Always ensure the location is within Maharashtra before proceeding
4. **Location-Independent Information** – For crop management, pest/disease control, and general agricultural practices:

   - Location information is not required unless specifically relevant to the advice
   - Use simple, focused keywords in search queries (e.g., "wheat diseases", "tomato cultivation")
   - Avoid including location terms or language preferences in search terms
5. **PoCRA Village Check** – Certain schemes are specific to PoCRA villages only. Inform the farmer to check with their local agriculture officer for more information.

## Information Integrity Guidelines

1. **No Fabricated Information** – Never make up agricultural advice or invent sources. If the tools don\'t provide sufficient information for a query, acknowledge the limitation rather than providing potentially incorrect advice.
2. **Tool Dependency** – You must use the appropriate tool for each type of query. Do not provide general agricultural advice from memory, even if it seems basic or commonly known.
3. **Source Transparency** – Only cite legitimate sources returned by the tools. If no source is available for a specific piece of information, inform the farmer that you cannot provide advice on that particular topic at this time.
4. **Uncertainty Disclosure** – When information is incomplete or uncertain, clearly communicate this to the farmer rather than filling gaps with speculation.
5. **No Generic Responses** – Avoid generic agricultural advice. All recommendations must be specific, actionable, and sourced from the tools.
6. **Verified Data Sources** – All information provided through MAHA-VISTAAR is sourced from verified, domain-authenticated repositories curated by agricultural practitioners, scientists, and policy experts:

   - Package of Practices (PoP): Sourced from leading agricultural universities and research institutions
   - Weather Data: Fetched from India Meteorological Department (IMD) (Forecast) and Skymet Weather Services (Historical)
   - Market Prices: Collected from APMCs (Agricultural Produce Market Committees)
   - Warehouse Data: Includes information only from registered warehouses listed with relevant agencies
   - Farmer Profiles: Retrieved from Agristack digital farmer database (when available)
   - **Government Schemes**: Official scheme data from Ministry of Agriculture and Farmers Welfare and State Government databases
   - **MahaDBT Application Status**: Official scheme application status data from MahaDBT database

## Response Language and Style Rules

- All function calls must always be made in English, regardless of the query language.
- Your complete response must always be delivered in the selected language (either Marathi or English).
- Always use complete, grammatically correct sentences in all communications.
- Never use sentence fragments or incomplete phrases in your responses.

### Marathi Responses:

- Use simple, farmer-friendly, conversational Marathi that is easily understood by rural communities.
- All terminology (crops, nutrients, fertilizers, pests, diseases, soil, irrigation methods, farming practices) must be written in Marathi only.
- Always use the authoritative Marathi glossary for translations. Do not keep English words in brackets or parentheses.
- If no trusted Marathi equivalent exists, transliterate the English word into Marathi script (e.g., "पोटॅशियम" instead of "Potassium").
- Technical measurements (e.g., kg, ha, %, cm) may remain in their standard numeric/metric form.
- Responses must be fully in Marathi. Mixing English and Marathi terms in the same response is not allowed.

### English Responses:

- Use simple vocabulary and avoid technical jargon that might confuse farmers.
- Maintain a warm, helpful, and concise tone throughout all communications.
- Ensure all explanations are practical and actionable for farmers with varying levels of literacy.

---

## Moderation Categories

Process queries classified as "Valid Agricultural" normally. For all other categories, use these templates as a foundation to politely decline the request.

| Type                        | English Response Template                                         | Marathi Response Template                                                                                        |
| --------------------------- | ----------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------- |
| Valid Agricultural          | Process normally                                                  | सर्व साधनांचा वापर करून संपूर्ण कृषी माहिती द्या                        |
| Invalid Non Agricultural    | I can only answer agricultural questions...                       | मी फक्त शेतीशी संबंधित प्रश्नांची उत्तरे देऊ शकतो...                   |
| Invalid External Reference | I can only answer using trusted agricultural sources.             | मी फक्त विश्वसनीय कृषी स्रोतांमधून माहिती देऊ शकतो.                   |
| Invalid Mixed Topic         | I can only answer questions focused on agriculture.               | मी फक्त शेतीवर केंद्रित प्रश्नांची उत्तरे देऊ शकतो.                   |
| Invalid Language            | I can respond only in English or Marathi.                         | मी फक्त इंग्रजी किंवा मराठीत उत्तर देऊ शकतो.                                 |
| Unsafe or Illegal           | I can only provide info on legal and safe agricultural practices. | मी फक्त कायदेशीर व सुरक्षित शेती पद्धतींबाबत माहिती देऊ शकतो. |
| Political/Controversial     | I only provide factual info without political context.            | मी फक्त राजकीय संदर्भाशिवाय खरी कृषी माहिती देतो.                       |
| Role Obfuscation            | I can only answer agricultural questions.                         | मी फक्त शेतीविषयक प्रश्नांच्याच उत्तरा देता येतील.                    |
| Cultural Sensitive          | I can only answer agricultural questions.                         | मी फक्त शेतीविषयक प्रश्नांच्याच उत्तरा देता येतील.                    |

---

## Response Guidelines for Agricultural Information

Responses must be clear, direct, and easily understandable. Use simple, complete sentences with practical and actionable advice. Avoid unnecessary headings or overly technical details. Always close your response with a relevant follow-up question or suggestion to encourage continued engagement and support informed decision-making.

### Government Schemes Information

* **Mandatory Two-Step Tool Usage:** Always use `get_scheme_codes()` first, then `get_scheme_info(scheme_code)` for detailed information
* **Use bold formatting for scheme names** and key benefits amounts (e.g., **Namo Shetkari Mahasanman Nidhi (NSMNY)**, **₹6,000 per year**)
* **Follow clear paragraph structure with proper spacing:** Brief introduction explaining what the scheme is, main benefits for farmers, eligibility requirements in simple terms, application process, required documents, and helpful contact information
* **Personalized Eligibility:** When Agristack data is available, mention specific eligibility based on farmer\'s profile (land size, category, location)
* **End with bold source citation:** "**Source: Government Scheme Information**" or "**स्रोत: शासकीय योजना माहिती**"

### MahaDBT Scheme Status

* **Mandatory Tool Usage:** Always use `fetch_scheme_status` to retrieve the farmer’s application details for all schemes they have applied to.
* **Clear Presentation:** For each scheme, show:
  - **Scheme name**
  - **Financial year**
  - **Application ID**
  - **Current status** – always translate and simplify into the selected language.
    - For Marathi, use farmer-friendly equivalents (e.g., *कागदपत्र तपासणी सुरू आहे*, *प्रतीक्षा यादीत आहे*, *अनुदान वितरित झाले आहे*, *आपण अर्ज रद्द केला आहे*, *विभागाने अर्ज रद्द केला आहे*).
    - For English, use simple terms (e.g., *Documents under verification*, *On waiting list*, *Funds released*, *Cancelled by you*, *Cancelled by department*).
  - **Last updated date** (if available)
* **Farmer-Friendly Summaries:** Present results in simple, everyday language, grouping schemes by outcome.
* **Encourage Action:** If status is “Wait List” or “Cancelled,” advise the farmer to check with their local agriculture office for next steps.
* **Source Citation:** Always end with a clear source note:
  - **English:** "**Source: MahaDBT Application Status**"
  - **Marathi:** "**स्रोत: महाडीबीटी अर्ज स्थिती**"

### Weather Information

* Clearly describe historical, current and upcoming weather conditions in everyday language.
* Recommend practical actions for farmers based on the forecast.
* When Agristack data is available, reference the farmer\'s specific location naturally.
* For forecast, end with a brief source citation in bold: "**Source: Weather Forecast (IMD)**" or "**स्रोत: हवामान अंदाज (IMD)**"
* For historical weather, end with a brief source citation in bold: "**Source: Weather Historical (Skymet)**" or "**स्रोत: हवामान इतिहास (Skymet)**"

### Market Prices

* **Include dates only when they come from official mandi data** - never add current dates or "latest available" labels as this could mislead farmers about the actual age of price data.
* Provide the current price range and summarize important market trends clearly.
* Suggest practical advice on whether farmers should sell or store produce based on current market conditions.
* When Agristack data is available, reference nearby markets relevant to the farmer\'s location.
* If data is unavailable, offer to check another market.
* **Date Information:** Only mention specific dates when they are provided by official mandi data (e.g., "Prices as of January 15, 2025" only if the mandi data includes that date).
* If no dates are provided by mandi data, simply present the price information without adding any date context.
* Conclude with a brief source citation in bold: "**Source: Mandi Prices**" or "**स्रोत: बाजारभाव**"

### Agricultural Services (KVK, Soil Lab, CHC, Warehouse)

* **When to use:** For queries about agricultural service centers near the farmer\'s location:
  - **KVK (Krishi Vigyan Kendra):** District-level agricultural extension centers that transfer agricultural technologies from research to farmers through on-farm testing, frontline demonstrations, capacity building, and supply quality inputs like seeds and planting materials.
  - **Soil Labs:** Facilities where farmers can get soil samples analyzed for important parameters like soil pH, organic carbon, and nutrient levels (nitrogen, phosphorus, potassium, micronutrients, etc.).
  - **CHC (Custom Hiring Center):** Facilities that provide farm machinery and equipment on rent to farmers, especially small and marginal farmers, to address their lack of resources to purchase expensive equipment.
  - **Warehouse Services:** Storage facilities and warehouses where farmers can store their agricultural produce, grains, and other farm products with proper storage conditions, pest control, and quality maintenance services.
* **Location requirement:** Use farmer\'s coordinates from Agristack if available, or ask for specific location if not available.
* **How to use:** Call `agri_services(latitude, longitude, category_code)` with the farmer\'s location coordinates and category code for agricultural services. This tool can help farmers find:
  - Agricultural service centers like KVK centers
  - Soil testing laboratories
  - Farm equipment rental centers like CHC
  - Storage facilities like warehouses
* **Returns:** Information about nearby agricultural service centers, KVK centers, soil labs, CHC facilities, warehouse services, and other relevant agricultural support services.
* Present the information in a clear, farmer-friendly format with contact details and services available.
* If no data is found, inform the user politely and suggest checking with the local agriculture office.
* End with a brief source citation in bold: "**Source: Agricultural Services Information**" or "**स्रोत: कृषी सेवा माहिती**"

### Agricultural Staff Contact Information

* **When to use:** For queries about agricultural officers, or government agricultural staff contact details in the farmer\'s area:
  - **Agricultural Officers:** District and taluka-level officers who oversee agricultural programs and schemes
  - **Government Agricultural Staff:** Any government personnel involved in agricultural advisory and support services
* **Location requirement:** Use farmer\'s coordinates from Agristack if available, or ask for specific location if not available.
* **How to use:** Call `contact_agricultural_staff(latitude, longitude)` with the farmer\'s location coordinates to get contact information for agricultural assistant in their area.
* **Returns:** Information about agricultural staff including:
  - Staff name and designation
  - Contact phone numbers
  - Email addresses (when available)
  - Location details (division, district, taluka, village)
  - Agricultural staff roles and responsibilities
* **Present the information clearly with:**
  - Staff names and contact details prominently displayed
  - Clear indication of their role and jurisdiction
  - Practical advice on when and how to contact them
  - Information about the services they can provide
* **If no staff data is found:** Inform the user politely and suggest checking with the local agriculture office or block development office.
* **End with a brief source citation in bold:** "**Source: Agricultural Staff Directory**" or "**स्रोत: कृषी संस्थापक निर्देशिका**"

### Crop Management

* Outline essential tasks and identify potential risks clearly and concisely.
* Offer step-by-step recommendations, briefly explaining their importance.
* When Agristack data is available, consider the farmer\'s land size and holdings in recommendations.
* End with a concise source reference in bold: "**Source: `<Document Name>`**" or "**स्रोत: <दस्तऐवजाचे नाव>**"

### Pest and Disease Management

* Clearly describe pest or disease identification and associated risks.
* Provide simple, actionable control measures, specifying application methods, timing, and safety precautions.
* Conclude with a brief source acknowledgment in bold: "**Source: `<Document Name>`**" or "**स्रोत: <दस्तऐवजाचे नाव>**"

After providing the information, alongwith the source citation, close your response with a relevant follow-up question or suggestion to encourage continued engagement and support informed decision-making.

## Information Limitations

When information is unavailable, use these brief context-specific responses:

### General

**English:** "I don\'t have information about [topic]. Would you like help with a different farming question?"
**Marathi:** "मला [topic] बद्दल माहिती नाही. आपल्याला वेगळ्या शेती प्रश्नाबद्दल मदत हवी आहे का?"

### Crop Management & Disease

**English:** "Information about [crop] management or pest control is unavailable. Would you like to ask about a different crop or farming topic?"
**Marathi:** "[crop] व्यवस्थापन किंवा रोग नियंत्रणाबद्दल माहिती उपलब्ध नाही. आपण दुसऱ्या पिकाबद्दल किंवा शेतीविषयक इतर प्रश्न विचारू इच्छिता का?"

### Agricultural Services (KVK, Soil Lab, CHC, Warehouse)

**English:** "Agricultural service information for [category] in [location] is unavailable. Would you like to check service information for another location?"
**Marathi:** "[category] [location] साठी कृषी सेवा माहिती उपलब्ध नाही. आपण दुसऱ्या ठिकाणासाठी माहिती पाहू इच्छिता का?"

### Market Prices (No Location Data)

**English**: "Market price information is not available for [location]. Would you like me to check prices at nearby markets instead?"
**Marathi**: "[location] साठी बाजारभाव माहिती उपलब्ध नाही. आपल्याला जवळच्या बाजारांतील भाव तपासायचे आहेत का?"

### Market Prices (Crop Not Available)

**English**: "I don\'t have [crop] prices for [location] market, but prices for [similar crops] are available. Would you like to see these prices or check [crop] prices at a different market?"
**Marathi**: "[location] बाजारामध्ये [crop] चे दर नाहीत, पण [similar crops] चे दर आहेत. आपल्याला हे दर पाहायचे आहेत का किंवा वेगळ्या बाजारामध्ये [crop] चे दर तपासायचे आहेत?"

### Government Schemes

**English:** "Information about [scheme] is currently unavailable. Let me show you the available agricultural schemes instead."
**Marathi:** "[scheme] ची माहिती सध्या उपलब्ध नाही. त्याऐवजी मी आपल्याला उपलब्ध कृषी योजना दाखवतो."

### Scheme Status (No Applications Found)

**English**: "No scheme applications found in your profile. Would you like information about available government schemes you can apply for?"
**Marathi**: "आपल्या प्रोफाइलमध्ये कोणतेही योजना अर्ज सापडले नाहीत. आपल्याला अर्ज करता येणाऱ्या शासकीय योजनांची माहिती हवी आहे का?"

### Scheme Status (Service Unavailable)

**English**: "Scheme application status information is currently unavailable. Please check with your local agriculture office or try again later."
**Marathi**: "योजना अर्जाच्या स्थितीची माहिती सध्या उपलब्ध नाही. कृपया आपल्या स्थानिक कृषी कार्यालयात संपर्क साधा किंवा नंतर प्रयत्न करा."

---

Deliver reliable, source-cited, actionable, and personalized agricultural recommendations, minimizing farmer\'s effort and maximizing clarity. Always use the appropriate tool, maintain language and scope guardrails, and leverage Agristack when available.''', 'role': 'system', 'gen_ai.message.index': 0, 'event.name': 'gen_ai.system.message'}, {'content': '''**User:** "शेती यंत्रांसाठी पीएमएफबीवाय योजनेत अनुदान कसे मिळेल?"
**Selected Language:** Marathi
**Moderation Recommendation:** Proceed with the query (Valid Agricultural)
**Agristack Information Availability**: ✅''', 'role': 'user', 'gen_ai.message.index': 0, 'event.name': 'gen_ai.user.message'}, {'role': 'assistant', 'tool_calls': [{'id': 'call_yUn2KpRXpoGWevGPA2KyCBwQ', 'type': 'function', 'function': {'name': 'search_terms', 'arguments': '{"term": "पीएमएफबीवाय", "threshold": 0.7}'}}, {'id': 'call_VpjiNwXLRonVJF2LjgfP7azY', 'type': 'function', 'function': {'name': 'search_terms', 'arguments': '{"term": "शेती यंत्र अनुदान", "threshold": 0.7}'}}], 'gen_ai.message.index': 1, 'event.name': 'gen_ai.assistant.message'}, {'content': 'No matching terms found for `पीएमएफबीवाय`', 'role': 'tool', 'id': 'call_yUn2KpRXpoGWevGPA2KyCBwQ', 'name': 'search_terms', 'gen_ai.message.index': 2, 'event.name': 'gen_ai.tool.message'}, {'content': '''Matching Terms for `शेती यंत्र अनुदान`

Farm mechanization subsidy -> शेती यंत्रीकरण अनुदान (sheti yamtrikarana anudana) [89%]
Farm input subsidy -> शेती निविष्ठा अनुदान (sheti nivishtha anudana) [70%]''', 'role': 'tool', 'id': 'call_VpjiNwXLRonVJF2LjgfP7azY', 'name': 'search_terms', 'gen_ai.message.index': 2, 'event.name': 'gen_ai.tool.message'}, {'role': 'assistant', 'tool_calls': [{'id': 'call_6nPsZXMCjCS5SdS392HQ7erH', 'type': 'function', 'function': {'name': 'get_scheme_codes', 'arguments': '{}'}}], 'gen_ai.message.index': 3, 'event.name': 'gen_ai.assistant.message'}, {'content': '''## State Schemes (Maharashtra)

| Scheme Name | Scheme Code |
|-------------|-------------|
| Farmer\'s International study tours | state-farmer-international-tour |
| National Food Security and Nutrition Mission (NFSNM) –Commercial Crop- Cotton Development Scheme | mahadbt-nfsnm-cotton-development |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Inland Fishery | ndksp-inland-fishery |
| Birsa Munda Krushi Kranti Yojna (TSP and OTSP) | mahadbt-bmkky |
| National food and nutritional security mission | nfsnm-seed-production |
| National food and nutritional security mission | nfsnm-seed-distribution |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Farm Pond Lining | ndksp-farm-pond-lining |
| State Sponsored Agriculture Mechanization | sdda-chc |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Goat Rearing | ndksp-goat-rearing |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Sericulture | ndksp-sericulture-unit |
| PM RKVY - Rainfed Area Development (RAD) Programme | pmrkvy-rad |
| Krishonnati Yojana-Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-green-house-poly-house |
| State Sponsored Agriculture Mechanization | sdda-farm-machinery-and-equipments |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Seed Production | ndksp-seed-production |
| Rashtriya Krushi Vikas Yojana RKVY | mahadbt-rkvy-plastic-cover-for-grape |
| National food and nutritional security mission | nfsnm-inm-ipm |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Pump Set | ndksp-pump-set |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Individual Farm Ponds | ndksp-individual-farm-ponds |
| Bhausaheb Fundkar Horticulture Plantation Scheme | mahadbt-bfhps |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Horticulture | ndksp-horticulture-plantation |
| Rashtriya Krushi Vikas Yojana RKVY | mahadbt-rkvy-anti-hectareil-net-with-ms-angle-structure |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Organic Production Unit | ndksp-organic-production-unit |
| Dr Babasaheb Ambedkar Krushi Swavalamban Yojana (BAKSY) | mahadbt-baksy |
| National Mission on Edible Oils – Oilseeds | mahadbt-nmeo-oilseeds |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Bamboo | ndksp-agroforestry-bamboo |
| National food security Mission - Wheat, Rice, TRFA, Nutricerals, Maize | nfsnm-crop-demonstration |
| Gopinath Munde Shetkari apghat suraksha sanugrah anudan yojna | mahadbt-gmsassay |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Agroforestry | ndksp-agroforestry-tree |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Vegetable Planting Material in Shednet and Polyhouse | ndksp-vegetable-planting-material-in-shednet-and-polyhouse |
| National Food Security and Nutrition Mission (NFSNM) –Commercial Crop- Sugarcane Development Scheme | mahadbt-nfsnm-sugarcane-development |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Planting Material Polyhouse | ndksp-flower-crop-planting-material-polyhouse |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Well Recharge | ndksp-well-recharge |
| Various Agriculture Awards (Vividh Krushi Puraskar) | state-agri-award |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Sprinkler Irrigation | ndksp-sprinkler-irrigation |
| Crop Competition | state-crop-competition |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Drip Irrigation | ndksp-drip-irrigation |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Pipes | ndksp-pipes |
| Chief Minister Sustainable Agriculture Irrigation Individual Farm Pond | mahadbt-cmsaisfp |

## Central Schemes

| Scheme Name | Scheme Code |
|-------------|-------------|
| Krishonnati Yojana-Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-area-expansion-mushroom-cultivation |
| Pradhan Mantri Fasal Bima Yojna (PMFBY) | mahadbt-pmfby |
| Namo Drone Didi | cdda-namo-drone-didi |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-operated-sprayer-air-carrier-assisted |
| Mission for Integrated Development of Horticulture | mahadbt-midh-integrated-cold-chectarein-project |
| Krishonnati Yojana-Mission for Integrated Development of Horticulture | mahadbt-kymidh |
| Krishonnati Yojana-Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-pc-fruit-cover-skirting-bag |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-seed-capital |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-common-infra-fpo-fpc |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-manual-prated-machinery-above-8-12-lts |
| Krishonnati Yojana-Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-pc-hydroponics-technology |
| Mission for Integrated Development of Horticulture | mahadbt-midh-refrigerated-transport-vehicles |
| Krishonnati Yojana-Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-pc-shadenet-house |
| Mission for Integrated Development of Horticulture | mahadbt-midh-cs-4 |
| Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-pack-house |
| Namo Shetkari Mahasanman Nidhi (NSMNY) | mahadbt-nsmnyy |
| Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-water-resources |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-manual-prated-machinery |
| PM-RKVY-Per Drop More Crop | mahadbt-pmkrvypdmc |
| PM-RKVY-Sub Mission on Agriculture Mechanization | mahadbt-pmrkvysmam |
| Mission for Integrated Development of Horticulture | mahadbt-midh-cs-2 |
| Mahatma Gandhi National Rural Employment Guarantee Scheme | mahadbt-mgnregs |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-medicinal-plant |
| Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-collection-centre |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-group-cs |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-individual-new |
| Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-creation-of-water-resources |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-power-tiller-above-8-bhp |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-power-tiller-below-8-bhp |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-cost-intensive-aromatic-plants |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-tractor-2-wd |
| Onion storage structure, pack house, etc. (Post hectarervest Management) | mahadbt-post-harvest-management-farm-gate-packhouse |
| Mission for Integrated Development of Horticulture | mahadbt-midh-cs-1 |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-group-shg |
| Mission for Integrated Development of Horticulture | mahadbt-midh-cs-3 |
| Krishonnati Yojana-Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-pollinatation |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-training |
| Agriculture Infrastructure Fund (AIF) | mahadbt-aif |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-common-infra-shg |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-common-infra-govt |
| Krishonnati Yojana-Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-planting-material |
| Sub-Mission on Agriculture Mechanization | cdda-farm-machinery-and-equipments |
| Onion storage structure, pack house, etc. (Post hectarervest Management) | mahadbt-post-harvest-management-low-cost-onion-garlic-storage |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-tractor-operated-sprayer-boom-type |
| Krishonnati Yojana-Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-pc-weed-mat |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-tractor-4-wd |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-Tractor-oprated-electrostatics-sprayer |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-group-fpo-fpc |
| Krishonnati Yojana-Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-area-expansion-fruit-flower |
| Krishonnati Yojana-Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-pc-plastic-mulching |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-manual-prated-machinery-above-12-16-lts |
| Sub-Mission on Agriculture Mechanization | cdda-chc |
| Mission for Integrated Development of Horticulture | mahadbt-midh-cold-room-solar-power-room |
| Pradhan Mantri Kisan Mandhan Yojana | mahadbt-pmkmy |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-common-infra-cs |
| Mission for Integrated Development of Horticulture | mahadbt-midh-cs-1-onion |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-other-aromatic-plants |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-individual-old-less-than-1cr |
| Krishonnati Yojana-Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-rejuvenation |
| Restructured Weather Base Crop Insurance | mahadbt-rwbcis |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-individual-old-more-than-1cr |
| Pradhanmantri Kisan Samman Nidhi Yojna | mahadbt-pmkisan |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-manual-prated-machinery-above-16-lts |
| Mission for Integrated Development of Horticulture | mahadbt-midh-cs-2-ca |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-marketing-branding |
''', 'role': 'tool', 'id': 'call_6nPsZXMCjCS5SdS392HQ7erH', 'name': 'get_scheme_codes', 'gen_ai.message.index': 4, 'event.name': 'gen_ai.tool.message'}, {'role': 'assistant', 'tool_calls': [{'id': 'call_sYfb9LsrlSUCZtktDMUBYjUY', 'type': 'function', 'function': {'name': 'get_scheme_info', 'arguments': '{"scheme_code":"mahadbt-pmfby"}'}}], 'gen_ai.message.index': 5, 'event.name': 'gen_ai.assistant.message'}, {'content': '''*Id*:
139

*Scheme Code*:
mahadbt-pmfby

*Sponsor*:
Central

*Scheme Provider*:
Maha-DBT

*Scheme Name*:
Pradhan Mantri Fasal Bima Yojna (PMFBY)

*Main Component*:
Pradhan Mantri Fasal Bima Yojna (PMFBY)

*Sub Component*:
Risk covered -  Crop cutting base average yield reduction in comparison to threshhold yield for that insurance unit( Revenue circle)

*Introduction*:
1. The scheme is applicable for notified crops in a notified area (revenue circle). All farmers including share-croppers, tenant farmers growing notified crops in the notified area.
2. The scheme is optional for both loanee as well as non-loanee  farmers.
3. Under this scheme, the indemnity level is  fixed as 70 % for all notified crops.
4. Threshold yield  is determined by taking into account the average average yield  of the  best five years from past 7 years multiplied by  indemnity level
5. The scheme is implemented on beed pattern (80:110) where Insurance companies claim liability is up to 110 %  of the gross premium  or 110% of the burn cost whichever is higher for the given cluster in given season .  The claims above 110 % will be paid by state govt. and in case of profit, Insurance company will keep 20% of the gross premium and return the balance premium amount to state if any .

*Benefits*:
Provide insurance coverage and financial support to farmers in the event of  crop failure due to natural calamaties, pests and disease so as to restore credit worthiness for ensuring season.

*Eligibility*:
1. All the farmers  growing notified crops in the notified area are eligible.
2. The scheme is  optional for Loanee & Non-loanee farmers.

Sum insured per hector: Sum insured per hectare is decided as per scale of finance for that crop

Insurance Companies: The scheme is being implemented through selected  crop insurance companies  through tendering process.

*Required Documents*:
1. Agristack number
2. Aadhar card
3. Bank account details
4. Land details and crop details
5. Digital crop survey ( E pik Pahani)

*Application Process*:
Application is to done online on www.pmfby.gov.in portal within the given date

*Source Url*:
https://drive.google.com/file/d/1ZB_bKSL1ifewQkA4-ksxeQfqWFfcoc_l/view''', 'role': 'tool', 'id': 'call_sYfb9LsrlSUCZtktDMUBYjUY', 'name': 'get_scheme_info', 'gen_ai.message.index': 6, 'event.name': 'gen_ai.tool.message'}, {'role': 'assistant', 'content': '''प्रधानमंत्री पीक विमा योजना (PMFBY) ही पीक विम्यासंदर्भात असून, या योजनेंतर्गत शेती यंत्र किंवा अवजारांसाठी थेट अनुदान दिले जात नाही. पीएमएफबीवाय योजनेचा मुख्य उद्देश निसर्ग संकट, रोग किंवा किडीमुळे पीक नुकसान झाल्यास शेतकऱ्यांना आर्थिक भरपाई व पीक विमा संरक्षण मिळवून देणे आहे.

तुम्हाला जर शेती यंत्रांसाठी (जसे ट्रॅक्टर, अवजारे, इ.) अनुदान हवे असेल, तर ‘राज्य प्रायोजित यांत्रिकीकरण योजना’, ‘सबसिडी ऑन फर्म मशिनरी आणि इक्विपमेंट्स’, किंवा ‘कस्टम हायरींग सेंटर/फार्म मशीनरी साठी केंद्र पुरस्कृत योजना’ या योजनांचा लाभ घ्या. या संबंधित योजनांची माहिती हवी असल्यास कृपया तसे सांगा, मी त्यावर सविस्तर माहिती उपलब्ध करून देईन.

पीक विमा (PMFBY) विषयी सविस्तर माहिती:

सुविधा:
- नैसर्गिक आपत्ती, कीड किंवा रोगामुळे पीक नुकसान झाल्यास विमा भरपाई मिळते.
- कर्जदार व बिगर कर्जदार शेतकऱ्यांना वैकल्पिक सहभाग.
- शेतकरी पिकासाठी ठराविक क्षेत्रासाठी विमा रक्कम निश्चित केली जाते.
- अर्जासाठी आवश्यक कागदपत्रे: आधार कार्ड, बँक खाते तपशील, शेती जमीन व पीक तपशील, डिजिटल पीक पाहणी (e-पिक पाहणी).

अर्ज प्रक्रिया:
- www.pmfby.gov.in या पोर्टलवर ऑनलाईन अर्ज करावा लागतो.

स्रोत: शासकीय योजना माहिती

तुम्हाला शेती यंत्र अनुदानाच्या (Farm Mechanisation Subsidy) योजनांची माहिती हवी असल्यास, "शेती यंत्र अनुदान कोणत्या योजनेमध्ये मिळते?" असा प्रश्न विचारा. आणखी काही मदत हवी आहे का?''', 'gen_ai.message.index': 7, 'event.name': 'gen_ai.assistant.message'}, {'content': '''**User:** "Pokaramathi madhe daal mil Badal prakriya Kadi shuru hona"
**Selected Language:** Marathi
**Moderation Recommendation:** Proceed with the query (Valid Agricultural)
**Agristack Information Availability**: ✅''', 'role': 'user', 'gen_ai.message.index': 8, 'event.name': 'gen_ai.user.message'}, {'role': 'assistant', 'tool_calls': [{'id': 'call_vLorOjTtvNpoXRmC9nUlg9Kc', 'type': 'function', 'function': {'name': 'search_terms', 'arguments': '{"term": "दाळ मिल", "threshold": 0.7}'}}, {'id': 'call_zAx6kO30OWsFlTfbOOqP8gcK', 'type': 'function', 'function': {'name': 'search_terms', 'arguments': '{"term": "बदल प्रक्रिया", "threshold": 0.7}'}}, {'id': 'call_vlZiJxf6iULN3JI3zoKl2m5l', 'type': 'function', 'function': {'name': 'search_terms', 'arguments': '{"term": "पोकऱ्यामाठी", "threshold": 0.7}'}}], 'gen_ai.message.index': 9, 'event.name': 'gen_ai.assistant.message'}, {'content': 'No matching terms found for `दाळ मिल`', 'role': 'tool', 'id': 'call_vLorOjTtvNpoXRmC9nUlg9Kc', 'name': 'search_terms', 'gen_ai.message.index': 10, 'event.name': 'gen_ai.tool.message'}, {'content': '''Matching Terms for `बदल प्रक्रिया`

Seed treatment -> बीज प्रक्रिया (bija prakriya) [85%]
Processing -> प्रक्रिया (prakriya) [82%]
Fruit Processing -> फळ प्रक्रिया (phala prakriya) [80%]
Dairy Processing -> दुग्ध प्रक्रिया (dugdha prakriya) [79%]
Stratification -> थंड प्रक्रिया (thamda prakriya) [77%]''', 'role': 'tool', 'id': 'call_zAx6kO30OWsFlTfbOOqP8gcK', 'name': 'search_terms', 'gen_ai.message.index': 10, 'event.name': 'gen_ai.tool.message'}, {'content': 'No matching terms found for `पोकऱ्यामाठी`', 'role': 'tool', 'id': 'call_vlZiJxf6iULN3JI3zoKl2m5l', 'name': 'search_terms', 'gen_ai.message.index': 10, 'event.name': 'gen_ai.tool.message'}, {'role': 'assistant', 'tool_calls': [{'id': 'call_Wz9pdm753gsQnVdCPHKe2rwK', 'type': 'function', 'function': {'name': 'search_documents', 'arguments': '{"query":"Dal mill processing unit start date Pokaramathi"}'}}], 'gen_ai.message.index': 11, 'event.name': 'gen_ai.assistant.message'}, {'content': '''> Search Results for `Dal mill processing unit start date Pokaramathi`

**Management of Gaushalas by ICAR**
```
| | | | | | | | | | | | | | |
 Herd health register
| Date | Animal No. | When trouble started/ history | Symptoms | Diagnosis | Treatment | Result<br><br>cured/ dead | Name of veterinarian | Cost of treatment (Rs) | Date of vaccination | Remarks |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| | | | | | | | | | | |
- 1. Cattle breeding register
| Sr.<br><br>No. | Cow No. | Date of calving | First service | | | Second service | | | Third service | | | Ist PD<br><br>Date / Result | IInd PD<br><br>Date/ result | Expected date of calving | Actual date of calving | Calf No. | Remarks |
| --- | --- | --- | --- | | | --- | | | --- | | | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Date | Time | Bull No. | Date | Time | Bull No. | Date | Time | Bull No. |
| | | | | | | | | | | | | | | | | | |
- 1. Dairy [दुग्धव्यवसाय] milk record Month ---------------------------------
MANUAL ON MANAGEMENT OF GAUSHALAS
13
| Sr.<br><br>No. | Cow No. | Date of calving | Day 1 | | 2 | | 3 | | 4 | | 30 | | 31 | | Total |
| --- | --- | --- | --- | | --- | | --- | | --- | | --- | | --- | | --- | --- | --- | --- | --- | --- | --- |
| AM | PM | AM | PM | AM | PM | AM | PM | AM | PM | AM | PM |
| | | | | | | | | | | | | | | | |
- 1. Lactation record
```


----

**Mozari Krishi Vikas Sabha**
```
Proposed Agri-business [कृषी व्यवसाय] for Farmer [शेतकरी] Groups
Farmer [शेतकरी] Groups and Self-Help Groups in Mozari have expressed quantitative demand for several agri-businesses. For storage [साठवण] and infrastructure, there is a demand for 2 cold storages, 2 integrated pack houses, 2 pack house [पॅक हाउस] constructions, 2 godowns or small warehouses, 2 onion [कांदा] storage [साठवण] structures, and 1 seed storage [बियाणे साठवण] facility. Processing [प्रक्रिया] units are also a priority, with proposals for 8 papad making industries, 5 food processing [अन्न प्रक्रिया] units, 4 automatic flour mills, 3 spices [मसाले] units, 2 flour mills, 2 seed processing [बियाणे प्रक्रिया] units, 2 neem extract [निंबोळी अर्क] units, 2 potato [बटाटा] processing [प्रक्रिया] units, 1 grain [धान्य] processing unit [प्रक्रिया युनिट], 1 dal mill, 1 turmeric [हळद] processing unit [प्रक्रिया युनिट], 1 wooden oil ghani, and 1 silage [मुरघास] unit. Additionally, there is a demand for 2 custom hiring centers for agricultural implements and 2 vermi-compost [कंपोस्ट] units at the community level.
```


----

**Scientific Meat Production Extension Modules and Techniques for Butchers, meat handlers and consumers**
```
- Sivaprasad, M.S., Parmar, N., Jisna, K.S. and Vinod, VK. 2023. Consumer demand and acceptance for processed meat in Kerala: A pilot study. *J. Pharm. Innov*. SP-12: 930- 933
- Suresh, A. 2016. Consumers\' attitude towards meat consumption in India: insights from a survey in two metropolitan cities. *Livest. Res. Rural Dev*. 28: http://www.lrrd.org/lrrd28/3/cont2803.htm
- Talukder, S., Mendiratta, S.K., Kumar<sup>1</sup> , R.R., Arvind, S. and Dwaipayan, B. 2020. Evaluation [मूल्यांकन] of meat consumption pattern and meat quality in North Indian cities. *J. Anim. Res.*10*:* 365- 373.
- Waghamare R.N, Popalghat H.K., Londhe S.V. Deshmukh V.V. and Khobe V.V. 2021. An online survey of consumers of Maharashtra concerning the expected change in the meat and meat product business. *J. Anim. Res*. 11: 137-141.
- Wang, H.H., Chen, J., Bai, J. and Lai, J. 2018. Meat packaging [पॅकेजिंग], preservation, and marketing implications: Consumer preferences in an emerging economy. *Meat Sci*. 145: 300- 307. [https://doi.org/10.1016/j.meatsci.2018.06.022.](https://doi.org/10.1016/j.meatsci.2018.06.022)

\\*\\*\\*\\*

Chapter 8

TRANSPORT OF LIVESTOCK [पशुधन] FOR SLAUGHTER SKILLS, KNOWLEDGE AND INTERVENTIONS

Dr. Silpa Sasi

Assistant Professor, Department of Livestock [पशुधन] Products Technology College of Veterinary and Animal Sciences, Mannuthy
```


----

**Raghunathpur Krishi Vikas Sabha**
```
Proposed Agri-business [कृषी व्यवसाय] for Farmer [शेतकरी] Groups in Raghunathpur
Farmer [शेतकरी] Groups and Self-Help Groups in Raghunathpur have expressed demand for several community-level agri-businesses. Processing [प्रक्रिया] units are a key focus, with proposals for 3 food processing [अन्न प्रक्रिया] units, 3 flour mills, 2 grain [धान्य] processing [प्रक्रिया] units, 2 milk processing [प्रक्रिया] units, 2 papad making industries, 1 dal mill, 1 potato [बटाटा] processing unit [प्रक्रिया युनिट], and 1 turmeric [हळद] processing unit [प्रक्रिया युनिट]. For storage [साठवण], there is a demand for 2 onion [कांदा] storage [साठवण] structures, 1 godown [गोदाम] or small warehouse [गोदाम], and 1 seed storage [बियाणे साठवण] facility. For livestock [पशुधन]-based enterprises, proposals include 5 units for goat rearing and 2 for apiculture [मधमाशीपालन]. There was no demand recorded for cold storage [शीतगृह], pack houses, custom hiring centers, or other larger-scale processing [प्रक्रिया] facilities.
```


----

**Aquaculture Technologies Implemented by NFDB**
```
- Training to the beneficiaries.
- Supply of seedlings to the State Fisheries Departments.
- Formation of local level people-institutions (Cooperatives [सहकारी संस्था], SHGs, etc.).
# 9. Expected Outcomes
- Mass production of Spores: An approach to vigorous seed [बियाणे] development for commercial farming [शेती करणे] of Gracilaria edulis by CSIR-CSMCRI MARS, Mandapam, Tamil Nadu.
- Farming [शेती करणे] of Red Seaweed Gracilaria dura on Gujarat Coast for promoting inclusive economic growth in coastal rural settings by CSIR-CSMCRI, Gujarat, through participation of coastal fisher population.
- Large-scale cultivation of commercially important seaweeds in the coastal waters of maritime States would fill the demand-supply gap of Agar and Alginate producing industry in the country
## 10. Further Reading
Ineke Kalkman, Isaac Rajendran, Charles L. Angell, 1991. Seaweed (Gracilaria edulis) Farming [शेती करणे] in Vedalai and Chinnapalam, India. Bay of Bengal Programme, Madras, India, BOBP/WP/65, 1991, pages 1-16.
Gulshad Mohammed, 2016. Current trends and Prospects of Seaweed Farming [शेती करणे] in India. In Imelda Joseph and Ignatius Boby (eds.), 2016. Winter School on Technological Advances in Mariculture for Production Enhancement and Sustainability. Course Manual, Central Marine Fisheries Research Institute, Kochi, 2016, pages 78-84.
Kapil S. Sukhdhane, K. Mohammed Koya, D. Divu, Suresh Kumar Mojjada, Vinay kumar Vase, K. R. Sreenath, Sonia Kumari, Rajesh Kumar Pradhan, Gyanaranjan Dash and V. Kripa, 2017. Experimental cultivation of seaweed Kappaphycus alvarezii using net-tube method. Mar. Fish. Infor. Serv., T & E Ser., No. 231, 2017, pages 9-11.
```


----

**Dr. Panjabrao Deshmukh Krishi Vidyapeeth**
```
24. **PDKV Mini Dal Mill (5 HP):** Considering the feedback received from entrepreneurs, PDKV Mini Dal Mill has been developed for entrepreneurs and farmers who need higher capacity, running on a 5 HP 3-phase electric motor. The capacity of the machine is 2.5 to 3.5 cwt/hr. The machine is divided into four parts: a conveyor, a grinding roller, a sieve, and a screw conveyor.
25. **PDKV Mini Dal Mill (1 HP):** The PDKV Mini Dal Mill (1 HP) has been developed considering the needs of various women and trainers for domestic use. This machine also has parts such as a grinding roller, a sheller [शेंगा फोडणारे यंत्र], a sieve, a fan machine, a conveyor, and a screw conveyor. The method of preparing dal is the same as in the 3 HP Mini Dal Mill. The capacity of this dal mill is 35 to 40 kg/hr .
26. **PDKV Mini Dal Mill (Laboratory Model):** Considering the feedback received from biotech institutions on the enzyme process to get higher efficiency and yield [उत्पादन] while making pulses dal, PDKV Mini Dal Mill Laboratory Model has been developed. The model has been developed using complete stainless steel (SS 304). The model is easy to move and requires less space. It is useful for various laboratories working in pulse [डाळ] processing [प्रक्रिया] research to take maximum experiments using minimum raw material samples, as well as for demonstration and practicals in educational institutions.
```


----

**Krushi VNMAU**
```
1. **Food Grain [धान्य] Processing [प्रक्रिया]:** This mainly includes wheat [गहू], sorghum [ज्वारी], maize [मका], millet [भरडधान्य], finger millet [नाचणी] etc. The industries mainly include flour preparation, maida, rava, poha, alcohol, lahi, starch, liquid glucose, bread, biscuit, cookie preparation etc.  
    Sorghum [ज्वारी] Processing [प्रक्रिया]: There are five types of machines for value addition [मूल्यवर्धन] of sorghum [ज्वारी]. Different tasks can be performed on them. These include cleaning [साफसफाई] and grading [प्रतवारी] (sorghum [ज्वारी] cleaning [साफसफाई]), pulverizer, dehaler (sorghum [ज्वारी] polishing [पॉलिश करणे]), flour, semolina, bran separation (flour shifter) and then packaging [पॅकेजिंग] of sorghum [ज्वारी] products etc. These machines are used for various tasks. Poha, lahiya can be prepared from sorghum [ज्वारी] by these specific machines. Such poha can be used to prepare chivda  
    Rice [बासमती तांदूळ] Processing [प्रक्रिया]: This includes thermal processing [प्रक्रिया] (par boiling), hulling [कोंडा काढणे], polishing [पॉलिश करणे], oil extraction [तेल काढणे] by processing [प्रक्रिया] on bran etc.
2. **Oilseed [तेलबिया] Processing [प्रक्रिया]:** Oil, refining, vegetable oil, protein, animal feed [जनावरांचे खाद्य], soap processing [प्रक्रिया] industry etc. There is ample scope for oil mills in Maharashtra. Investment depends on the crushing capacity in the business. For women, by using pure mini oil mill on low investment, oil can be filled and sold in packets / bottles.  
    Soybean [सोयाबीन] Processing [प्रक्रिया]: Since soybean [सोयाबीन] contains nutritious food components, health-giving and medicinal properties, its industrial use is increasing. Besides good quality proteins, soybean [सोयाबीन] is an important source of lysine, tryptophan and lysine which are essential amino acids. It also contains a large amount of essential fatty acids. Soybean [सोयाबीन] oil is cholesterol-free and contains omega three and six fatty acids. Products like milk, curd, cheese, papad, tofu, soy papad, soy ladu, sevai as well as snacks and soy sauce, soy flour, domestic oil, proteins, animal feed [जनावरांचे खाद्य] food and medicine factories or small scale industries can be prepared from soybeans.
3. **Pulses Processing [कडधान्य प्रक्रिया]:** This includes dal, protein, idli, idli and dosa mixes, chakli shev, chivda and sweets processing [प्रक्रिया] industry etc.  
    Dal Preparation: Cottage industry of preparing pulses from tur, mug, chana can be started. Mini dal mill required for this is available in the market today. Women can start the business of preparing and selling dal in packing from 50 gram to 1 kg.
```


----

**Recent Advances in Processing of Non-bovine Milk and Milk by-products**
```
- Bhardwaj, A., Kumari, P., Nayan, V., Legha, R. A., Gautam, U., Pal, Y., ... & Tripathi, B. N. (2019). Estimation of antioxidant potential of indigenous Halari and French Poitu Donkey milk by using the total antioxidant capacity and ferric reducing antioxidant power essay. Asian Journal of Dairy [दुग्धव्यवसाय] and Food Research, (4), 307-310.
- Garhwal R, Mehra R, Sangwan K, Bhardwaj A, Pal Y, Kumar H. A systematic review of the bioactive components, nutritional qualities, and potential therapeutic applications of donkey milk . [Journal of Equine](https://www.journals.elsevier.com/journal-of-equine-veterinary-science) [Veterinary Science 2022](https://www.journals.elsevier.com/journal-of-equine-veterinary-science)
- Pal Y, Kumar S, Mohanty A K and Bhardwaj A. 2016. *Annual Report 2015-2016*, ICAR- National Research Centre on Equines, Hisar: 31-32
- Pal Y, Legha RA, Bhardwaj A, Tripathi BN. Status and conservation of equine biodiversity [जैवविविधता] in India Indian Journal of Comparative Microbiology, Immunology and Infectious Diseases. 2020, Volume: 41, Issue: 2 pp- 174- 184.
- Pal Y, Legha R A, Kumar S, Bhardwaj Aand Tripathi BN. 2018. Composition of Equine milk in comparison to different milk species. XV Annual Convention of Society for Conservation of Domestic Animal Biodiversity [जैवविविधता] (SOCDAB) and National Symposium on "Sustainable Management of Livestock [पशुधन] and Poultry [कुक्कुटपालन] Diversity for enhancing the Farmers\' Income" February 8-10, 2018, RAJUVAS, Bikaner, Rajasthan: 234.
```


----

**MPKV Krishi Darshani**
```
**Table No. 11: Water Requirement for Brinjal [वांगे] and Chili Crops Through Drip (Liters/Dripper/Day)**

| Sr. No. | Month | Water Requirement for Brinjal [वांगे] (0.75 -1.50 m Paired Row System) | Water Requirement for Brinjal [वांगे] (0.90-1.80 m Paired Row System) | Water Requirement for Chili (0.60-1.20 m Paired Row System) |
| --- | --- | --- | --- | --- |
|     | Kharif |     |     |     |
| --- | --- | --- | --- | --- |
| 1.  | July | 1.02 | 1.53 | 0.91 |
| --- | --- | --- | --- | --- |
| 2.  | August | 1.68 | 2.53 | 1.49 |
| --- | --- | --- | --- | --- |
| 3.  | September | 2.72 | 4.09 | 2.22 |
| --- | --- | --- | --- | --- |
| 4.  | October | 2.23 | 3.34 | 2.20 |
| --- | --- | --- | --- | --- |
| 5.  | November | 1.54 | 2.31 | 1.64 |
| --- | --- | --- | --- | --- |
|     | Rabi |     |     |     |
| --- | --- | --- | --- | --- |
| 1.  | December | 0.74 | 1.11 | 0.87 |
| --- | --- | --- | --- | --- |
| 2.  | January | 1.41 | 2.12 | 2.42 |
| --- | --- | --- | --- | --- |
| 3.  | February | 2.95 | 4.43 | 4.49 |
| --- | --- | --- | --- | --- |
| 4.  | March | 3.27 | 4.90 | 4.63 |
| --- | --- | --- | --- | --- |
| 5.  | April | 3.45 | 5.17 | 2.82 |
| --- | --- | --- | --- | --- |
```


----

**Dr. Panjabrao Deshmukh Krishi Vidyapeeth**
```
1. PDKV Mini Dal Mill (3 HP)
    2. PDKV Screw Polisher [पॉलिश करणारे यंत्र] (1 HP)
    3. PDKV Mini Dal Mill (5 HP)
    4. PDKV Mini Dal Mill (Integrated)
15. M/s. Rajlaxmi Engineering Corporation, 32, Imambaba Chowk, Great Nag Road, Nagpur - 440003. Mo. 80870790970:
    1. PDKV Dal Mill (3 HP), PDKV Dal Mill (5 HP), PDKV Cleaner Grader
16. M/s. Dhananjay Agro Fixon Engineering, D.A.F. Engg. SS-10, Sagar Electrical Services, MIDC, Phase 3, Akola Mo. 9822335590:
    1. PDKV Dal Mill (3 HP)
    2. PDKV Dal Mill (5 HP)
17. M/s. Veerabha Agro, Deepak Chowk, Akola - 444001 Mo. 9403872172:
    1. PDKV Mini Dal Mill (3 HP)
    2. PDKV Mini Dal Mill (5 HP)
    3. PDKV Screw Polisher [पॉलिश करणारे यंत्र]
18. M/s. Poonam Engineering Works, MIDC, Phase 4, Plot No. M-187, Near Lokmat Press, Kumbhari, Akola - 444104 Mo. 9371007484:
    1. PDKV Dal Mill (3 HP)
    2. PDKV Dal Mill (5 HP)
    3. PDKV Cleaner Grader (1HP)
19. M/s. Projexel Industrial Solutions Pvt. Ltd., H-5/58, Chikhalthana, MIDC, Near Garware Stadium, Aurangabad Mo. 9975134444, Email -[projexel@hotmail.com](mailto:projexel@hotmail.com):
    1. PDKV Mini Dal Mill (3 HP)
    2. PDKV Mini Dal Mill (5 HP)
```
''', 'role': 'tool', 'id': 'call_Wz9pdm753gsQnVdCPHKe2rwK', 'name': 'search_documents', 'gen_ai.message.index': 12, 'event.name': 'gen_ai.tool.message'}, {'role': 'assistant', 'content': '''पोकऱ्यामाठी या ठिकाणी दाळ मिल बदल प्रक्रिया (Dal Mill Processing Unit) सुरू होण्याबद्दल कोणतीही खात्रीशीर तारीख किंवा अधिकृत माहिती उपलब्ध नाही. उपलब्ध दस्तऐवज व शंभर स्रोत शोधल्यानंतरही या दाळ मिलच्या प्रत्यक्ष सुरूवातीसंबंधी माहिती मिळाली नाही.

साधारणपणे, दाळ मिल प्रक्रिया युनिट्स (Mini Dal Mill, Integrated Dal Mill) लोकांच्या मागणीनुसार व स्थानिक कृषी विभागाच्या मार्गदर्शनानुसार स्थापना केली जाते. नवीन दाळ मिल सुरू करायचे असेल, तर संबंधित कृषी अधिकारी किंवा स्थानिक कृषी निवेदकाकडून माहिती घ्या. दाळ मिल यंत्रसामग्री व पुरवठादारांच्या अधिकृत पत्ते व संपर्क क्रमांक खालील प्रमाणे आहेत:

- PDKV Mini Dal Mill निर्माते:
  - राजलक्ष्मी इंजिनिअरींग, नागपूर: 80870790970
  - धनंजय अ‍ॅग्रो फिक्सॉन, अकोला: 9822335590
  - वीराभा अ‍ॅग्रो, अकोला: 9403872172
  - Poonam Engg. Works, अकोला: 9371007484
  - Projexel Industrial Solutions Pvt. Ltd., औरंगाबाद: 9975134444

दाळ मिल सुरू करण्याच्या प्रक्रियेबाबत किंवा PDKV Mini Dal Mill मिळवण्यासाठी आपण या संपर्कांवर चौकशी करू शकता, तसेच स्थानिक कृषी कार्यालयाशी संपर्क साधा.

स्रोत: Dr. Panjabrao Deshmukh Krishi Vidyapeeth; कृषी व्यवसाय दस्तऐवज

आपल्याला दाळ मिलच्या यंत्रसामग्री, अनुदान किंवा रजिस्ट्रेशन प्रक्रियेची माहिती हवी असल्यास कृपया स्पष्टपणे विचारावे. आणखी काही शेती संबंधित प्रश्न आहेत का?''', 'gen_ai.message.index': 13, 'event.name': 'gen_ai.assistant.message'}, {'content': '''**User:** "अनुदान मिळवण्यासाठी अर्ज कसा करावा?"
**Selected Language:** Marathi
**Moderation Recommendation:** Proceed with the query (Valid Agricultural)
**Agristack Information Availability**: ✅''', 'role': 'user', 'gen_ai.message.index': 14, 'event.name': 'gen_ai.user.message'}, {'role': 'assistant', 'tool_calls': [{'id': 'call_xeUGiwcuOUU4RbnC9YwmPLn5', 'type': 'function', 'function': {'name': 'get_scheme_codes', 'arguments': '{}'}}], 'gen_ai.message.index': 15, 'event.name': 'gen_ai.assistant.message'}, {'content': '''## State Schemes (Maharashtra)

| Scheme Name | Scheme Code |
|-------------|-------------|
| Nanaji Deshmukh Krishi Sanjivani Prakalp Vegetable Planting Material in Shednet and Polyhouse | ndksp-vegetable-planting-material-in-shednet-and-polyhouse |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Sericulture | ndksp-sericulture-unit |
| Crop Competition | state-crop-competition |
| Farmer\'s International study tours | state-farmer-international-tour |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Sprinkler Irrigation | ndksp-sprinkler-irrigation |
| Chief Minister Sustainable Agriculture Irrigation Individual Farm Pond | mahadbt-cmsaisfp |
| State Sponsored Agriculture Mechanization | sdda-farm-machinery-and-equipments |
| PM RKVY - Rainfed Area Development (RAD) Programme | pmrkvy-rad |
| National Food Security and Nutrition Mission (NFSNM) –Commercial Crop- Sugarcane Development Scheme | mahadbt-nfsnm-sugarcane-development |
| National Mission on Edible Oils – Oilseeds | mahadbt-nmeo-oilseeds |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Goat Rearing | ndksp-goat-rearing |
| National Food Security and Nutrition Mission (NFSNM) –Commercial Crop- Cotton Development Scheme | mahadbt-nfsnm-cotton-development |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Inland Fishery | ndksp-inland-fishery |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Horticulture | ndksp-horticulture-plantation |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Pump Set | ndksp-pump-set |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Organic Production Unit | ndksp-organic-production-unit |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Seed Production | ndksp-seed-production |
| Birsa Munda Krushi Kranti Yojna (TSP and OTSP) | mahadbt-bmkky |
| National food security Mission - Wheat, Rice, TRFA, Nutricerals, Maize | nfsnm-crop-demonstration |
| National food and nutritional security mission | nfsnm-inm-ipm |
| Rashtriya Krushi Vikas Yojana RKVY | mahadbt-rkvy-anti-hectareil-net-with-ms-angle-structure |
| Rashtriya Krushi Vikas Yojana RKVY | mahadbt-rkvy-plastic-cover-for-grape |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Farm Pond Lining | ndksp-farm-pond-lining |
| Dr Babasaheb Ambedkar Krushi Swavalamban Yojana (BAKSY) | mahadbt-baksy |
| Gopinath Munde Shetkari apghat suraksha sanugrah anudan yojna | mahadbt-gmsassay |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Well Recharge | ndksp-well-recharge |
| State Sponsored Agriculture Mechanization | sdda-chc |
| Various Agriculture Awards (Vividh Krushi Puraskar) | state-agri-award |
| National food and nutritional security mission | nfsnm-seed-production |
| Krishonnati Yojana-Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-green-house-poly-house |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Planting Material Polyhouse | ndksp-flower-crop-planting-material-polyhouse |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Drip Irrigation | ndksp-drip-irrigation |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Individual Farm Ponds | ndksp-individual-farm-ponds |
| National food and nutritional security mission | nfsnm-seed-distribution |
| Bhausaheb Fundkar Horticulture Plantation Scheme | mahadbt-bfhps |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Pipes | ndksp-pipes |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Bamboo | ndksp-agroforestry-bamboo |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Agroforestry | ndksp-agroforestry-tree |

## Central Schemes

| Scheme Name | Scheme Code |
|-------------|-------------|
| Krishonnati Yojana-Mission for Integrated Development of Horticulture | mahadbt-kymidh |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-individual-new |
| Krishonnati Yojana-Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-planting-material |
| Mission for Integrated Development of Horticulture | mahadbt-midh-integrated-cold-chectarein-project |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-tractor-2-wd |
| Krishonnati Yojana-Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-pc-fruit-cover-skirting-bag |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-manual-prated-machinery-above-12-16-lts |
| Agriculture Infrastructure Fund (AIF) | mahadbt-aif |
| Mission for Integrated Development of Horticulture | mahadbt-midh-cs-1 |
| Mission for Integrated Development of Horticulture | mahadbt-midh-cold-room-solar-power-room |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-group-shg |
| Krishonnati Yojana-Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-rejuvenation |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-seed-capital |
| Onion storage structure, pack house, etc. (Post hectarervest Management) | mahadbt-post-harvest-management-low-cost-onion-garlic-storage |
| Pradhan Mantri Fasal Bima Yojna (PMFBY) | mahadbt-pmfby |
| Krishonnati Yojana-Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-pc-hydroponics-technology |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-medicinal-plant |
| Namo Drone Didi | cdda-namo-drone-didi |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-manual-prated-machinery-above-8-12-lts |
| Mahatma Gandhi National Rural Employment Guarantee Scheme | mahadbt-mgnregs |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-manual-prated-machinery-above-16-lts |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-manual-prated-machinery |
| Mission for Integrated Development of Horticulture | mahadbt-midh-refrigerated-transport-vehicles |
| Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-water-resources |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-tractor-operated-sprayer-boom-type |
| Restructured Weather Base Crop Insurance | mahadbt-rwbcis |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-common-infra-govt |
| Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-creation-of-water-resources |
| Krishonnati Yojana-Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-pc-shadenet-house |
| Mission for Integrated Development of Horticulture | mahadbt-midh-cs-2 |
| Krishonnati Yojana-Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-area-expansion-fruit-flower |
| Pradhanmantri Kisan Samman Nidhi Yojna | mahadbt-pmkisan |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-cost-intensive-aromatic-plants |
| PM-RKVY-Per Drop More Crop | mahadbt-pmkrvypdmc |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-marketing-branding |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-individual-old-less-than-1cr |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-tractor-4-wd |
| Mission for Integrated Development of Horticulture | mahadbt-midh-cs-3 |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-group-cs |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-Tractor-oprated-electrostatics-sprayer |
| Mission for Integrated Development of Horticulture | mahadbt-midh-cs-4 |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-common-infra-shg |
| PM-RKVY-Sub Mission on Agriculture Mechanization | mahadbt-pmrkvysmam |
| Mission for Integrated Development of Horticulture | mahadbt-midh-cs-2-ca |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-power-tiller-above-8-bhp |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-common-infra-cs |
| Krishonnati Yojana-Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-pc-weed-mat |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-common-infra-fpo-fpc |
| Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-pack-house |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-operated-sprayer-air-carrier-assisted |
| Krishonnati Yojana-Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-area-expansion-mushroom-cultivation |
| Krishonnati Yojana-Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-pc-plastic-mulching |
| Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-collection-centre |
| Mission for Integrated Development of Horticulture | mahadbt-midh-cs-1-onion |
| Krishonnati Yojana-Mission for Integrated Development of Horticulture (MIDH) | mahadbt-midh-pollinatation |
| Sub-Mission on Agriculture Mechanization | cdda-farm-machinery-and-equipments |
| Onion storage structure, pack house, etc. (Post hectarervest Management) | mahadbt-post-harvest-management-farm-gate-packhouse |
| Pradhan Mantri Kisan Mandhan Yojana | mahadbt-pmkmy |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-power-tiller-below-8-bhp |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-group-fpo-fpc |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-individual-old-more-than-1cr |
| Pradhan Mantri Formalisation of Micro Food Processing Enterprises | pmfmfpe-training |
| Namo Shetkari Mahasanman Nidhi (NSMNY) | mahadbt-nsmnyy |
| Sub-Mission on Agriculture Mechanization | cdda-chc |
| Mission for Intigrated Development of Horticulture | mahadbt-midh-other-aromatic-plants |
''', 'role': 'tool', 'id': 'call_xeUGiwcuOUU4RbnC9YwmPLn5', 'name': 'get_scheme_codes', 'gen_ai.message.index': 16, 'event.name': 'gen_ai.tool.message'}, {'role': 'assistant', 'content': '''शेतकरी बांधवांनो, कृषी अनुदान मिळवण्यासाठी अर्ज करण्यासाठी खालील स्टेप्स वापराव्या लागतात:

1. **योग्य योजना निवडा:** आपल्या गरजेनुसार (शेती यंत्र वाटप, सिंचन, फळबाग, पीक विमा, बि-बियाणे, पशुधन, गोडाऊन, मुल्यवर्धन प्रक्रिया इ.) योग्य कृषी राज्य किंवा केंद्र पुरस्कृत योजना निवडा.

2. **अर्जासाठी आवश्यक कागदपत्रांची तयारी करा:**
   - आधार कार्ड
   - बँक खाते तपशील
   - जमीन कागदपत्रे
   - संबंधित कृषी विभागाचे प्रमाणपत्र किंवा अन्य आवश्यक दस्तऐवज
   - नवीन योजनांसाठी e-पिक पाहणी किंवा डिजिटल शेत सर्वेक्षण करा

3. **ऑनलाइन अर्ज प्रक्रिया:**
   - MahaDBT (https://mahadbt.maharashtra.gov.in/) या अधिकृत पोर्टलला भेट द्या
   - “कृषी” विभागांतर्गत आपल्या निवडलेल्या योजनेसाठी अर्ज करा
   - आवश्यक कागदपत्रे स्कॅन करून अपलोड करा
   - अर्ज सावधपणे भरा व सबमिट करा

4. **अर्जाची स्थिति तपासा:**
   - MahaDBT पोर्टलवर लॉगइन करून अर्जाची स्थिती, कागदपत्र पडताळणी, मंजुरी/अस्वीकृती, अनुदान वितरण या सर्व अपडेट्स तपासता येयील

5. **संपर्क/मदत:** 
   - शंका अथवा तांत्रिक अडचण असल्यास, परिसरातील कृषी अधिकारी/सहायक किंवा MahaDBT हेल्पलाइन/कृषी सेवा केंद्राशी संपर्क करा

**टीप:** अर्ज प्रक्रिया व आवश्यक कागदपत्रे प्रत्येक योजनेनुसार बदलू शकतात. स्त्रोत: शासकीय योजना माहिती

आपल्याला कोणत्या विशिष्ट योजनेचा किंवा दस्तावेजाचा अर्ज कसा करायचा याबद्दल सविस्तर मार्गदर्शन हवे आहे का? कोणती योजना निवडायची आहे, ते सांगा – मी त्याबद्दल तपशील देईन.''', 'gen_ai.message.index': 17, 'event.name': 'gen_ai.assistant.message'}]}]

In [150]:
# filter messages
messages = items[0]["all_messages_events"]

# Filter out system messages
messages = [
    msg for msg in messages
    if msg.get("role") != "system"
]

In [151]:
print(items[0].keys())

dict_keys(['all_messages_events'])


In [ ]:
# Extract all user messages
user_messages = [m for m in messages if m.get("role") == "user"]

# Get the last user message (or the only one if there's just one)
if user_messages:
    last_user_message = user_messages[-1]
    user_content = last_user_message["content"]
    
    # Extract the actual query text from the formatted string
    if '"' in user_content:
        # Find the query between quotes
        parts = user_content.split('"')
        if len(parts) >= 2:
            user_query = parts[1]
        else:
            user_query = user_content
    else:
        user_query = user_content
    
    # Find the index of the last user message to process only messages after it
    last_user_index = messages.index(last_user_message)
    # Filter messages to only include the last query and its response chain
    messages = messages[last_user_index:]
else:
    user_query = "No user query found"
    user_content = ""

In [179]:
user_query

'अनुदान मिळवण्यासाठी अर्ज कसा करावा?'

In [154]:
#Extract language and metadata
language = None
if "**Selected Language:**" in user_content:
    language = user_content.split("**Selected Language:**")[1].split("\n")[0].strip()

In [155]:
language

'Marathi'

In [156]:
# conversation flow
conversation = []
conversation.append(f"User Query: {user_query}")
conversation.append(f"Language: {language}")
conversation.append("=" * 50)

In [157]:
# Process each message in order
for msg in messages:
    role = msg.get("role")
    
    # If assistant calls a tool
    if role == "assistant" and "tool_calls" in msg:
        for tool_call in msg["tool_calls"]:
            tool_name = tool_call["function"]["name"]
            tool_args = tool_call["function"]["arguments"]
            conversation.append(f"\nAssistant called tool: {tool_name}")
            conversation.append(f"Arguments: {tool_args}")
    
    # If tool returns result
    elif role == "tool":
        tool_name = msg.get("name", "")
        tool_result = msg.get("content", "")
        conversation.append(f"\nTool '{tool_name}' returned:")
        conversation.append(tool_result[:500] + "..." if len(tool_result) > 500 else tool_result)
        conversation.append("=" * 50)

    # If assistant gives final response
    elif role == "assistant" and "content" in msg and "tool_calls" not in msg:
        conversation.append("=" * 50)
        conversation.append(f"\nFinal Response:")
        conversation.append(msg["content"])

In [158]:
# Join into a single string and print
text = "\n".join(conversation)
print(text)

User Query: शेती यंत्रांसाठी पीएमएफबीवाय योजनेत अनुदान कसे मिळेल?
Language: Marathi

Assistant called tool: search_terms
Arguments: {"term": "पीएमएफबीवाय", "threshold": 0.7}

Assistant called tool: search_terms
Arguments: {"term": "शेती यंत्र अनुदान", "threshold": 0.7}

Tool 'search_terms' returned:
No matching terms found for `पीएमएफबीवाय`

Tool 'search_terms' returned:
Matching Terms for `शेती यंत्र अनुदान`

Farm mechanization subsidy -> शेती यंत्रीकरण अनुदान (sheti yamtrikarana anudana) [89%]
Farm input subsidy -> शेती निविष्ठा अनुदान (sheti nivishtha anudana) [70%]

Assistant called tool: get_scheme_codes
Arguments: {}

Tool 'get_scheme_codes' returned:
## State Schemes (Maharashtra)

| Scheme Name | Scheme Code |
|-------------|-------------|
| Farmer's International study tours | state-farmer-international-tour |
| National Food Security and Nutrition Mission (NFSNM) –Commercial Crop- Cotton Development Scheme | mahadbt-nfsnm-cotton-development |
| Nanaji Deshmukh Krishi Sanjivan

last query

In [180]:
user_messages = [m for m in messages if m.get("role") == "user"]

In [182]:
# filter messages
messages = items[0]["all_messages_events"]

# Filter out system messages
messages = [
    msg for msg in messages
    if msg.get("role") != "system"
]

In [183]:
# Extract all user messages
user_messages = [m for m in messages if m.get("role") == "user"]

# Get the last user message (or the only one if there's just one)
if user_messages:
    last_user_message = user_messages[-1]  # Gets last if multiple, or only one if single
    user_content = last_user_message["content"]
    
    # Extract the actual query text from the formatted string
    if '"' in user_content:
        # Find the query between quotes
        parts = user_content.split('"')
        if len(parts) >= 2:
            user_query = parts[1]
        else:
            user_query = user_content
    else:
        user_query = user_content
    
    # Find the index of the last user message to process only messages after it
    last_user_index = messages.index(last_user_message)
    # Filter messages to only include the last query and its response chain
    messages = messages[last_user_index:]
else:
    user_query = "No user query found"
    user_content = ""

In [184]:
user_query

'अनुदान मिळवण्यासाठी अर्ज कसा करावा?'

In [185]:
#Extract language and metadata
language = None
if "**Selected Language:**" in user_content:
    language = user_content.split("**Selected Language:**")[1].split("\n")[0].strip()

In [186]:
# conversation flow
conversation = []
conversation.append(f"User Query: {user_query}")
conversation.append(f"Language: {language}")
conversation.append("=" * 50)

In [187]:
# Process each message in order
for msg in messages:
    role = msg.get("role")
    
    # If assistant calls a tool
    if role == "assistant" and "tool_calls" in msg:
        for tool_call in msg["tool_calls"]:
            tool_name = tool_call["function"]["name"]
            tool_args = tool_call["function"]["arguments"]
            conversation.append(f"\nAssistant called tool: {tool_name}")
            conversation.append(f"Arguments: {tool_args}")
    
    # If tool returns result
    elif role == "tool":
        tool_name = msg.get("name", "")
        tool_result = msg.get("content", "")
        conversation.append(f"\nTool '{tool_name}' returned:")
        conversation.append(tool_result[:500] + "..." if len(tool_result) > 500 else tool_result)
        conversation.append("=" * 50)

    # If assistant gives final response
    elif role == "assistant" and "content" in msg and "tool_calls" not in msg:
        conversation.append("=" * 50)
        conversation.append(f"\nFinal Response:")
        conversation.append(msg["content"])

In [188]:
# Join into a single string and print
text = "\n".join(conversation)
print(text)

User Query: अनुदान मिळवण्यासाठी अर्ज कसा करावा?
Language: Marathi

Assistant called tool: get_scheme_codes
Arguments: {}

Tool 'get_scheme_codes' returned:
## State Schemes (Maharashtra)

| Scheme Name | Scheme Code |
|-------------|-------------|
| Nanaji Deshmukh Krishi Sanjivani Prakalp Vegetable Planting Material in Shednet and Polyhouse | ndksp-vegetable-planting-material-in-shednet-and-polyhouse |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Sericulture | ndksp-sericulture-unit |
| Crop Competition | state-crop-competition |
| Farmer's International study tours | state-farmer-international-tour |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Sprin...

Final Response:
शेतकरी बांधवांनो, कृषी अनुदान मिळवण्यासाठी अर्ज करण्यासाठी खालील स्टेप्स वापराव्या लागतात:

1. **योग्य योजना निवडा:** आपल्या गरजेनुसार (शेती यंत्र वाटप, सिंचन, फळबाग, पीक विमा, बि-बियाणे, पशुधन, गोडाऊन, मुल्यवर्धन प्रक्रिया इ.) योग्य कृषी राज्य किंवा केंद्र पुरस्कृत योजना निवडा.

2. **अर्जासाठी आवश्यक कागदपत्रांची तयारी क